In [ ]:
!pip install transformers -U
!pip install gensim
import pandas as pd
data = pd.read_csv("/content/dataset.csv", engine="python")
data.head()

In [ ]:
data['text'] = data['text'].astype(str).fillna("")
data= data[data['text'].str.strip() != ""]
label_map = {"Positive": 0, "Negative": 1, "Neutral": 2}
data['decision'] = data['decision'].map(label_map)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import torch
from transformers import TrainingArguments, Trainer
from transformers import BertTokenizer, BertForSequenceClassification
# Load BERT tokenizer and model
tokenizer2 = BertTokenizer.from_pretrained('bert-base-uncased')
model2 = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

In [ ]:
X = list(data["text"])
y = list(data["decision"])
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2,stratify=y)


In [ ]:
from gensim.models import Word2Vec
sentences = [text.split() for text in X_train]
w2v_model = Word2Vec(
    sentences,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

In [ ]:
import numpy as np

def sentence_w2v(sentence, model, dim=300):
    words = sentence.split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

In [ ]:
X_train_w2v = np.array([sentence_w2v(s, w2v_model) for s in X_train])
X_val_w2v   = np.array([sentence_w2v(s, w2v_model) for s in X_val])

In [ ]:
class FusionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, w2v_embeddings, labels):
        self.encodings = encodings
        self.w2v = torch.tensor(w2v_embeddings, dtype=torch.float)
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["w2v"] = self.w2v[idx]
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
X_train_tokenized = tokenizer2(X_train, padding=True, truncation=True, max_length=512)
X_val_tokenized = tokenizer2(X_val, padding=True, truncation=True, max_length=512)

In [ ]:
train_dataset = FusionDataset(X_train_tokenized, X_train_w2v, y_train)
val_dataset   = FusionDataset(X_val_tokenized, X_val_w2v, y_val)

In [ ]:
import torch.nn as nn
from transformers import BertModel
class BertW2VEarlyFusion(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.bert = BertModel(config)
        self.dropout = nn.Dropout(0.3);

        self.classifier = nn.Linear(
            config.hidden_size + 300,   # 768 + 300
            config.num_labels
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        w2v=None,
        labels=None
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS]
        fused = torch.cat((cls_embedding, w2v), dim=1)
        fused = self.dropout(fused)

        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [ ]:
from transformers import BertConfig
from transformers import BertModel # Only need BertModel

config = BertConfig.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# Instantiate the custom model directly
model = BertW2VEarlyFusion(config)
bert_base_model = BertModel.from_pretrained("bert-base-uncased")
model.bert.load_state_dict(bert_base_model.state_dict())

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    # Convert labels to one-hot encoding for AUC calculation
    labels_one_hot = np.eye(len(np.unique(labels)))[labels]
    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "precision_weighted": precision_score(labels, preds, average="weighted"),
        "recall_weighted": recall_score(labels, preds, average="weighted"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "precision_macro": precision_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision_micro": precision_score(labels, preds, average="micro"),
        "recall_micro": recall_score(labels, preds, average="micro"),
        "f1_micro": f1_score(labels, preds, average="micro"),
    }

    # Add AUC only if there are enough classes and probabilities are available
    try:
        if logits.shape[1] > 2:
            # Multiclass AUC (One-vs-Rest)
            auc_score = roc_auc_score(labels_one_hot, logits, average='weighted', multi_class='ovr')
        else:
            # Binary AUC
            auc_score = roc_auc_score(labels, logits[:, 1]) # Assuming column 1 is positive class score
        metrics["auc"] = auc_score
    except ValueError as e:
        print(f"Could not compute AUC: {e}")
        # Handle cases where AUC cannot be computed (e.g., all labels are the same)

    return metrics

args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()
trainer.evaluate()

In [ ]:
all_sentences = [text.split() for text in X]
w2v_full_corpus = Word2Vec(
    all_sentences,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)
print("Word2Vec model trained on the full dataset (w2v_full_corpus).")

In [ ]:
from sklearn.model_selection import StratifiedKFold

all_fold_metrics = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold+1}/{skf.n_splits} ---")

    # a. Split X and y into train_index and val_index using the current fold's indices.
    X_train_fold = [X[i] for i in train_index]
    X_val_fold = [X[i] for i in val_index]
    y_train_fold = [y[i] for i in train_index]
    y_val_fold = [y[i] for i in val_index]

    # c. Generate Word2Vec embeddings for X_train_fold and X_val_fold
    X_train_w2v_fold = np.array([sentence_w2v(s, w2v_full_corpus) for s in X_train_fold])
    X_val_w2v_fold = np.array([sentence_w2v(s, w2v_full_corpus) for s in X_val_fold])

    # d. Tokenize X_train_fold and X_val_fold using tokenizer2
    X_train_tokenized_fold = tokenizer2(X_train_fold, padding=True, truncation=True, max_length=512)
    X_val_tokenized_fold = tokenizer2(X_val_fold, padding=True, truncation=True, max_length=512)

    # e. Create FusionDataset instances for the current train and validation sets
    train_dataset_fold = FusionDataset(X_train_tokenized_fold, X_train_w2v_fold, y_train_fold)
    val_dataset_fold = FusionDataset(X_val_tokenized_fold, X_val_w2v_fold, y_val_fold)

    # f. Re-initialize a new instance of BertW2VEarlyFusion and load pre-trained weights
    model_fold = BertW2VEarlyFusion(config)
    bert_base_model_for_fold = BertModel.from_pretrained("bert-base-uncased")
    model_fold.bert.load_state_dict(bert_base_model_for_fold.state_dict())

    # g. Create a new Trainer instance for the current fold
    # Re-using the 'args' and 'compute_metrics' defined previously
    trainer_fold = Trainer(
        model=model_fold,
        args=args,
        train_dataset=train_dataset_fold,
        eval_dataset=val_dataset_fold,
        compute_metrics=compute_metrics
    )

    # h. Train the model
    trainer_fold.train()

    # i. Evaluate the trained model and store metrics
    fold_metrics = trainer_fold.evaluate()
    all_fold_metrics.append(fold_metrics)

print("Cross-validation complete. All fold metrics collected.")

In [ ]:
aggregated_metrics = {}

# Aggregate metrics from all folds
for fold_metrics in all_fold_metrics:
    for metric_name, metric_value in fold_metrics.items():
        # Exclude non-numeric metrics or internal Trainer metrics like 'eval_runtime'
        if isinstance(metric_value, (int, float)) and not metric_name.startswith('eval_runtime') and not metric_name.startswith('eval_samples_per_second') and not metric_name.startswith('eval_steps_per_second') and not metric_name.startswith('epoch'):
            if metric_name not in aggregated_metrics:
                aggregated_metrics[metric_name] = []
            aggregated_metrics[metric_name].append(metric_value)

print("\n--- Aggregated Metrics Across All Folds ---")
for metric_name, values in aggregated_metrics.items():
    mean_value = np.mean(values)
    std_value = np.std(values)
    print(f"{metric_name}: Mean = {mean_value:.4f}, Std = {std_value:.4f}")

# Task
Define, train, and evaluate a BERT-only model using 5-fold stratified cross-validation, then aggregate its performance metrics. Following this, define, train, and evaluate a Word2Vec-only model using 5-fold stratified cross-validation and aggregate its metrics. Finally, compare the aggregated results of the BERT-only model, Word2Vec-only model, and the previously run BERT+Word2Vec fusion model to analyze their individual and combined contributions.

## Define BERT-only Model

### Subtask:
Define a new PyTorch model class, `BertOnlyClassifier`, that uses only BERT embeddings for classification. This will serve as a baseline for the BERT component.


**Reasoning**:
I need to define a new PyTorch model class, `BertOnlyClassifier`, that uses only BERT embeddings for classification, as specified in the subtask instructions. This involves importing necessary modules, defining the class, its constructor, and the forward pass to extract BERT embeddings and classify them.



In [ ]:
import torch.nn as nn
from transformers import BertModel

class BertOnlyClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BertModel(config)
        self.dropout = nn.Dropout(0.3) # Using the same dropout rate as in BertW2VEarlyFusion
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :] # [CLS] token embedding
        cls_embedding = self.dropout(cls_embedding)

        logits = self.classifier(cls_embedding)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

**Reasoning**:
Now that the `BertOnlyClassifier` model class is defined, I need to instantiate it, load the BERT configuration, and initialize its BERT component with pre-trained weights, similar to the setup for `BertW2VEarlyFusion`.



**Reasoning**:
The previous cell failed because the `config` variable was not defined within its scope. I need to re-initialize `config` before it is used to instantiate `BertOnlyClassifier`.



In [ ]:
from transformers import BertConfig, BertModel

# Re-using the config with num_labels=3
config = BertConfig.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# Instantiate the custom BertOnlyClassifier model
bert_only_model = BertOnlyClassifier(config)

# Load the state dictionary from a pre-trained BertModel to the custom model's bert component
bert_base_model_only = BertModel.from_pretrained("bert-base-uncased")
bert_only_model.bert.load_state_dict(bert_base_model_only.state_dict())

print("BertOnlyClassifier model initialized and pre-trained weights loaded.")

**Reasoning**:
Now that the `BertOnlyClassifier` is defined and initialized, I need to create a custom PyTorch `Dataset` class specifically for the BERT-only model. This dataset will handle the tokenized text inputs and the corresponding labels, preparing the data for training and evaluation with the `Trainer`.



In [ ]:
import torch

class BertOnlyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

print("BertOnlyDataset class defined.")

## Define Training Arguments and Metrics for BERT-only Model

### Subtask:
Re-use the `TrainingArguments` and `compute_metrics` function defined earlier for consistency in training and evaluation of the BERT-only model.

**Reasoning**:
Now that the `BertOnlyClassifier` and `BertOnlyDataset` are defined, and `TrainingArguments` and `compute_metrics` are ready for reuse, the next step is to implement the 5-fold stratified cross-validation to train and evaluate the BERT-only model. This involves iterating through each fold, preparing the data, initializing the model, training, and collecting evaluation metrics.



**Reasoning**:
The previous cell failed because `StratifiedKFold` was used without being imported. I need to explicitly import `StratifiedKFold` from `sklearn.model_selection` to resolve this `NameError`.



**Reasoning**:
The previous cell failed because `X` and `y` were not defined in its scope, leading to a `NameError`. I need to ensure `X` and `y` are properly initialized from the `data` DataFrame before being used in the `StratifiedKFold` split. I will also re-include the `StratifiedKFold` and `Trainer` imports as they were required and possibly not retained in the current execution context.



**Reasoning**:
The `NameError: name 'data' is not defined` indicates that the `data` DataFrame, essential for extracting `X` and `y`, is not available in the current scope. I need to re-execute the data loading and preprocessing steps that define the `data` DataFrame and its transformed columns (like 'text' and 'decision') before `X` and `y` are assigned. This ensures all necessary variables are present for the cross-validation loop to proceed.



**Reasoning**:
The previous cell failed with a `NameError` because `tokenizer2` was not defined within its scope. I need to re-initialize `tokenizer2` from `transformers` at the beginning of the cell to ensure it's available for tokenizing the text data during cross-validation.



**Reasoning**:
The previous cell failed with a `NameError` because `args` (TrainingArguments) was not defined within its scope. I need to re-initialize `args` and `compute_metrics` at the beginning of the cell to ensure they are available for the `Trainer` during cross-validation.



In [ ]:
import pandas as pd # Import pandas to read the CSV
from sklearn.model_selection import StratifiedKFold
from transformers import Trainer, BertTokenizer, TrainingArguments # Ensure Trainer, BertTokenizer, and TrainingArguments are imported
import numpy as np # Import numpy for compute_metrics
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
import torch
from transformers import BertModel # Ensure BertModel is imported for config

# Re-load and preprocess the data as it was not defined
data = pd.read_csv("/content/dataset.csv", engine="python")
data['text'] = data['text'].astype(str).fillna("")
data= data[data['text'].str.strip() != ""]
label_map = {"Positive": 0, "Negative": 1, "Neutral": 2}
data['decision'] = data['decision'].map(label_map)

# Ensure X and y are defined from the processed 'data' DataFrame
X = list(data["text"])
y = list(data["decision"])

# Re-initialize tokenizer2
tokenizer2 = BertTokenizer.from_pretrained('bert-base-uncased')

# Re-initialize config (from cell ac5a066d, if it wasn't retained)
config = BertConfig.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# Re-initialize compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    labels_one_hot = np.eye(len(np.unique(labels)))[labels]

    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "precision_weighted": precision_score(labels, preds, average="weighted"),
        "recall_weighted": recall_score(labels, preds, average="weighted"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "precision_macro": precision_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision_micro": precision_score(labels, preds, average="micro"),
        "recall_micro": recall_score(labels, preds, average="micro"),
        "f1_micro": f1_score(labels, preds, average="micro"),
    }

    try:
        if logits.shape[1] > 2:
            auc_score = roc_auc_score(labels_one_hot, logits, average='weighted', multi_class='ovr')
        else:
            auc_score = roc_auc_score(labels, logits[:, 1])
        metrics["auc"] = auc_score
    except ValueError as e:
        print(f"Could not compute AUC: {e}")

    return metrics

# Re-initialize args (TrainingArguments)
args = TrainingArguments(
    output_dir="./results_bert_only", # Changed output directory to avoid conflict
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs_bert_only', # Changed logging directory
    logging_steps=10,
)

# Ensure BertOnlyDataset and BertOnlyClassifier are defined
class BertOnlyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

class BertOnlyClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BertModel(config)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :] # [CLS] token embedding
        cls_embedding = self.dropout(cls_embedding)

        logits = self.classifier(cls_embedding)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}


all_bert_only_fold_metrics = []
skf_bert_only = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, val_index) in enumerate(skf_bert_only.split(X, y)):
    print(f"\n--- BERT-only Fold {fold+1}/{skf_bert_only.n_splits} ---")

    # a. Split X and y into train_index and val_index for the current fold
    X_train_fold = [X[i] for i in train_index]
    X_val_fold = [X[i] for i in val_index]
    y_train_fold = [y[i] for i in train_index]
    y_val_fold = [y[i] for i in val_index]

    # b. Tokenize X_train_fold and X_val_fold using tokenizer2
    X_train_tokenized_fold = tokenizer2(X_train_fold, padding=True, truncation=True, max_length=512)
    X_val_tokenized_fold = tokenizer2(X_val_fold, padding=True, truncation=True, max_length=512)

    # c. Create BertOnlyDataset instances for the current train and validation sets
    train_dataset_fold_bert_only = BertOnlyDataset(X_train_tokenized_fold, y_train_fold)
    val_dataset_fold_bert_only = BertOnlyDataset(X_val_tokenized_fold, y_val_fold)

    # d. Re-initialize a new instance of BertOnlyClassifier and load pre-trained weights
    # Make sure 'config' is available (it should be from previous cells)
    model_fold_bert_only = BertOnlyClassifier(config)
    bert_base_model_for_fold_bert_only = BertModel.from_pretrained("bert-base-uncased")
    model_fold_bert_only.bert.load_state_dict(bert_base_model_for_fold_bert_only.state_dict())

    # e. Create a new Trainer instance for the current fold
    # Re-using the 'args' and 'compute_metrics' defined previously
    trainer_fold_bert_only = Trainer(
        model=model_fold_bert_only,
        args=args,
        train_dataset=train_dataset_fold_bert_only,
        eval_dataset=val_dataset_fold_bert_only,
        compute_metrics=compute_metrics
    )

    # f. Train the model
    trainer_fold_bert_only.train()

    # g. Evaluate the trained model and store metrics
    fold_metrics_bert_only = trainer_fold_bert_only.evaluate()
    all_bert_only_fold_metrics.append(fold_metrics_bert_only)

print("BERT-only Cross-validation complete. All fold metrics collected.")

# Task
The code in the last cell is still running as it performs cross-validation, which can take some time. Once it completes, I will aggregate and print the mean and standard deviation of the performance metrics for the BERT-only model, as per the plan.

## Aggregate BERT-only Metrics

### Subtask:
Calculate and print the mean and standard deviation of the performance metrics collected during the 5-fold cross-validation for the BERT-only model.


**Reasoning**:
I need to aggregate the performance metrics collected for the BERT-only model across all 5 folds. This involves calculating the mean and standard deviation for each relevant metric and then printing the results to evaluate its overall performance, as specified in the subtask.



In [ ]:
import numpy as np

aggregated_bert_only_metrics = {}

# Aggregate metrics from all folds for the BERT-only model
for fold_metrics in all_bert_only_fold_metrics:
    for metric_name, metric_value in fold_metrics.items():
        # Exclude non-numeric metrics or internal Trainer metrics
        if isinstance(metric_value, (int, float)) and not metric_name.startswith('eval_runtime') and not metric_name.startswith('eval_samples_per_second') and not metric_name.startswith('eval_steps_per_second') and not metric_name.startswith('epoch'):
            if metric_name not in aggregated_bert_only_metrics:
                aggregated_bert_only_metrics[metric_name] = []
            aggregated_bert_only_metrics[metric_name].append(metric_value)

print("\n--- Aggregated Metrics Across All Folds for BERT-only Model ---")
for metric_name, values in aggregated_bert_only_metrics.items():
    mean_value = np.mean(values)
    std_value = np.std(values)
    print(f"{metric_name}: Mean = {mean_value:.4f}, Std = {std_value:.4f}")

## Define Word2Vec-only Model

### Subtask:
Define a new PyTorch model class, `Word2VecOnlyClassifier`, that uses only Word2Vec embeddings for classification. This model will take the averaged Word2Vec embeddings as input and pass them through a linear layer for classification. The model should include a dropout layer for regularization.

**Reasoning**:
I need to define the `Word2VecOnlyClassifier` class as specified, including its `__init__` and `forward` methods, to handle Word2Vec embeddings for classification.



In [ ]:
import torch.nn as nn
import torch

class Word2VecOnlyClassifier(nn.Module):
    def __init__(self, num_labels, embedding_dim=300):
        super().__init__()
        self.dropout = nn.Dropout(0.3) # Using the same dropout rate as before
        self.classifier = nn.Linear(embedding_dim, num_labels)

    def forward(
        self,
        w2v=None,
        labels=None
    ):
        # Word2Vec embeddings are already processed to be sentence vectors
        fused = self.dropout(w2v)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

print("Word2VecOnlyClassifier class defined.")

### Subtask:
Define a new PyTorch `Dataset` class, `Word2VecOnlyDataset`, that takes Word2Vec embeddings and labels as input. This dataset will be used to feed data to the `Word2VecOnlyClassifier` during training and evaluation.

**Reasoning**:
I need to define the `Word2VecOnlyDataset` class, which will encapsulate the Word2Vec embeddings and corresponding labels, making them compatible with PyTorch's `DataLoader` for training and evaluation.



In [ ]:
import torch

class Word2VecOnlyDataset(torch.utils.data.Dataset):
    def __init__(self, w2v_embeddings, labels):
        self.w2v = torch.tensor(w2v_embeddings, dtype=torch.float)
        self.labels = labels

    def __getitem__(self, idx):
        item = {"w2v": self.w2v[idx]}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

print("Word2VecOnlyDataset class defined.")

## Train and Evaluate Word2Vec-only Model with 5-Fold Cross-Validation

### Subtask:
Perform 5-fold stratified cross-validation to train and evaluate the `Word2VecOnlyClassifier` model. For each fold, create `Word2VecOnlyDataset` instances, initialize a new `Word2VecOnlyClassifier`, train it using the `Trainer`, and collect the evaluation metrics. Ensure to reuse `TrainingArguments` (`args`) and `compute_metrics` function as defined previously.

#### Instructions:
1. Initialize an empty list, `all_w2v_only_fold_metrics`, to store evaluation results for each fold.
2. Create a `StratifiedKFold` instance (`skf_w2v_only`) with `n_splits=5`, `shuffle=True`, and `random_state=42`.
3. Iterate through each fold generated by `skf_w2v_only.split(X, y)`.
4. Inside the loop, split `X` and `y` into `X_train_fold`, `X_val_fold`, `y_train_fold`, and `y_val_fold` for the current fold.
5. Generate Word2Vec embeddings for `X_train_fold` and `X_val_fold` using the `sentence_w2v` function and `w2v_full_corpus` (trained on the full dataset).
6. Create `Word2VecOnlyDataset` instances for the current train and validation sets using the generated Word2Vec embeddings and labels.
7. Initialize a new `Word2VecOnlyClassifier` instance for each fold, passing `num_labels=3` and `embedding_dim=300`.
8. Create a new `Trainer` instance for the current fold, passing the `Word2VecOnlyClassifier` model, `args`, the train and validation datasets, and `compute_metrics`.
9. Train the model using `trainer_fold_w2v_only.train()`.
10. Evaluate the trained model using `trainer_fold_w2v_only.evaluate()` and store the resulting metrics in `all_w2v_only_fold_metrics`.

**Reasoning**:
I will implement the 5-fold stratified cross-validation for the Word2Vec-only model as described in the previous subtask. This involves setting up the loop, preparing data for each fold, initializing the Word2Vec-only model, training it, and collecting evaluation metrics.



**Reasoning**:
The previous cell failed because the `sentence_w2v` function was not defined in the current scope. I need to re-include its definition within the cell to ensure it's available for generating Word2Vec embeddings during the cross-validation loop.



**Reasoning**:
The previous cell failed because `w2v_full_corpus` was not defined in the current scope. I need to re-initialize `w2v_full_corpus` from `gensim.models.Word2Vec` before the cross-validation loop to ensure it's available for generating Word2Vec embeddings.



In [ ]:
from sklearn.model_selection import StratifiedKFold
from transformers import Trainer
import numpy as np
import torch
from gensim.models import Word2Vec # Import Word2Vec

# Re-define sentence_w2v function as it was not in scope
def sentence_w2v(sentence, model, dim=300):
    words = sentence.split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

# Re-initialize w2v_full_corpus as it was not in scope (from cell 5b080deb)
# Make sure 'X' is defined globally or passed into this scope if necessary
# Assuming 'X' (list of text) is already defined from previous cells
all_sentences = [text.split() for text in X]
w2v_full_corpus = Word2Vec(
    all_sentences,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)
print("Word2Vec model trained on the full dataset (w2v_full_corpus) for Word2Vec-only model.")

all_w2v_only_fold_metrics = []
skf_w2v_only = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, val_index) in enumerate(skf_w2v_only.split(X, y)):
    print(f"\n--- Word2Vec-only Fold {fold+1}/{skf_w2v_only.n_splits} ---")

    # 4. Split X and y into train_index and val_index for the current fold
    X_train_fold = [X[i] for i in train_index]
    X_val_fold = [X[i] for i in val_index]
    y_train_fold = [y[i] for i in train_index]
    y_val_fold = [y[i] for i in val_index]

    # 5. Generate Word2Vec embeddings for X_train_fold and X_val_fold
    X_train_w2v_fold = np.array([sentence_w2v(s, w2v_full_corpus) for s in X_train_fold])
    X_val_w2v_fold = np.array([sentence_w2v(s, w2v_full_corpus) for s in X_val_fold])

    # 6. Create Word2VecOnlyDataset instances for the current train and validation sets
    train_dataset_fold_w2v_only = Word2VecOnlyDataset(X_train_w2v_fold, y_train_fold)
    val_dataset_fold_w2v_only = Word2VecOnlyDataset(X_val_w2v_fold, y_val_fold)

    # 7. Initialize a new Word2VecOnlyClassifier instance for each fold
    model_fold_w2v_only = Word2VecOnlyClassifier(num_labels=3, embedding_dim=300)

    # 8. Create a new Trainer instance for the current fold
    # Re-using the 'args' and 'compute_metrics' defined previously
    # Note: For Word2Vec-only model, input_ids and attention_mask are not used,
    # so we need a custom data_collator or ensure the Trainer handles it.
    # The current Trainer setup implicitly passes the dataset items as is.
    # The Word2VecOnlyClassifier expects 'w2v' and 'labels' which the dataset provides.
    trainer_fold_w2v_only = Trainer(
        model=model_fold_w2v_only,
        args=args, # Re-use args from previous BERT-only training, output_dir will be overwritten if not unique.
        train_dataset=train_dataset_fold_w2v_only,
        eval_dataset=val_dataset_fold_w2v_only,
        compute_metrics=compute_metrics
    )

    # 9. Train the model
    trainer_fold_w2v_only.train()

    # 10. Evaluate the trained model and store metrics
    fold_metrics_w2v_only = trainer_fold_w2v_only.evaluate()
    all_w2v_only_fold_metrics.append(fold_metrics_w2v_only)

print("Word2Vec-only Cross-validation complete. All fold metrics collected.")

**Reasoning**:
Now that the 5-fold cross-validation for the Word2Vec-only model is complete and its metrics are collected in `all_w2v_only_fold_metrics`, I need to aggregate these results by calculating the mean and standard deviation for each metric, as per the subtask instructions.



In [ ]:
import numpy as np

aggregated_w2v_only_metrics = {}

# Aggregate metrics from all folds for the Word2Vec-only model
for fold_metrics in all_w2v_only_fold_metrics:
    for metric_name, metric_value in fold_metrics.items():
        # Exclude non-numeric metrics or internal Trainer metrics
        if isinstance(metric_value, (int, float)) and not metric_name.startswith('eval_runtime') and not metric_name.startswith('eval_samples_per_second') and not metric_name.startswith('eval_steps_per_second') and not metric_name.startswith('epoch'):
            if metric_name not in aggregated_w2v_only_metrics:
                aggregated_w2v_only_metrics[metric_name] = []
            aggregated_w2v_only_metrics[metric_name].append(metric_value)

print("\n--- Aggregated Metrics Across All Folds for Word2Vec-only Model ---")
for metric_name, values in aggregated_w2v_only_metrics.items():
    mean_value = np.mean(values)
    std_value = np.std(values)
    print(f"{metric_name}: Mean = {mean_value:.4f}, Std = {std_value:.4f}")

**Reasoning**:
The subtask is to compare the aggregated results of the BERT-only model, Word2Vec-only model, and the previously run BERT+Word2Vec fusion model. I will now print a comparative summary of the mean metrics for all three models to facilitate analysis.



**Reasoning**:
The previous cell failed with a `NameError` because `aggregated_metrics` was not defined in the current scope. I need to re-initialize `aggregated_metrics` from the `all_fold_metrics` collected earlier for the BERT+Word2Vec fusion model before attempting to print the comparative analysis.



**Reasoning**:
The previous cell failed with a `NameError` because `aggregated_metrics` (for the BERT+Word2Vec fusion model) was not defined. This variable was supposed to contain the aggregated metrics from an earlier execution. I will re-create this dictionary with the mean values obtained from the prior successful execution of the BERT+Word2Vec fusion model's aggregation step (cell `3b69d0bd` in the notebook's history) to allow the comparative analysis to proceed.



In [ ]:
import numpy as np # Ensure numpy is imported for np.nan and np.mean

# Re-initialize aggregated_metrics for the BERT+W2V Fusion model with values from previous execution
aggregated_metrics = {
    'eval_loss': [0.6213],
    'eval_accuracy': [0.8647],
    'eval_precision_weighted': [0.8638],
    'eval_recall_weighted': [0.8647],
    'eval_f1_weighted': [0.8640],
    'eval_precision_macro': [0.8543],
    'eval_recall_macro': [0.8519],
    'eval_f1_macro': [0.8529],
    'eval_precision_micro': [0.8647],
    'eval_recall_micro': [0.8647],
    'eval_f1_micro': [0.8647],
    'eval_auc': [0.9557]
}

print("\n--- Comparative Analysis of Model Performance (Mean Metrics) ---")
print(f"{'Metric':<25} | {'BERT+W2V Fusion':<20} | {'BERT-only':<15} | {'Word2Vec-only':<15}")
print(f"{'-'*25} | {'-'*20} | {'-'*15} | {'-'*15}")

metrics_to_compare = [
    'eval_loss',
    'eval_accuracy',
    'eval_precision_weighted',
    'eval_recall_weighted',
    'eval_f1_weighted',
    'eval_auc'
]

for metric_name in metrics_to_compare:
    # Retrieve values for fusion model (already aggregated as means in this step)
    fusion_mean = aggregated_metrics.get(metric_name, [np.nan])[0] # Get the first (and only) element

    bert_only_mean = aggregated_bert_only_metrics.get(metric_name, [np.nan])
    bert_only_mean = np.mean(bert_only_mean) if bert_only_mean else np.nan

    w2v_only_mean = aggregated_w2v_only_metrics.get(metric_name, [np.nan])
    w2v_only_mean = np.mean(w2v_only_mean) if w2v_only_mean else np.nan

    print(f"{metric_name:<25} | {fusion_mean:<20.4f} | {bert_only_mean:<15.4f} | {w2v_only_mean:<15.4f}")

## Final Task

### Subtask:
Summarize the findings from comparing the BERT-only, Word2Vec-only, and BERT+Word2Vec fusion models, highlighting the individual contributions and the benefits of the fusion approach.


## Summary:

### Q&A
The findings from comparing the BERT-only, Word2Vec-only, and BERT+Word2Vec fusion models demonstrate that both BERT-based models significantly outperform the Word2Vec-only model. The BERT+Word2Vec fusion model generally achieves slightly better or comparable performance to the BERT-only model across most evaluation metrics, suggesting a marginal benefit from combining both embedding types.

### Data Analysis Key Findings
*   The **BERT-only model** exhibited strong performance with a mean `eval_accuracy` of 0.8637, a mean `eval_f1_weighted` of 0.8626, and a mean `eval_auc` of 0.9567, along with a mean `eval_loss` of 0.6302.
*   The **Word2Vec-only model** performed significantly worse across all metrics, achieving a mean `eval_accuracy` of only 0.3705, a mean `eval_f1_weighted` of 0.2145, and a mean `eval_auc` of 0.4658, with a much higher mean `eval_loss` of 1.0886.
*   The **BERT+Word2Vec Fusion model** demonstrated the best overall performance, with a mean `eval_accuracy` of 0.8647, a mean `eval_f1_weighted` of 0.8640, and the lowest mean `eval_loss` at 0.6213. Its mean `eval_auc` was 0.9557, which was marginally lower than the BERT-only model.
*   Comparing the BERT-based models, the fusion approach yielded a slight improvement in `eval_accuracy` (0.8647 vs. 0.8637) and `eval_f1_weighted` (0.8640 vs. 0.8626), and a lower `eval_loss` (0.6213 vs. 0.6302), indicating that Word2Vec embeddings, when combined with BERT, can offer a small but positive contribution.

### Insights or Next Steps
*   BERT's contextual embeddings are highly effective for this classification task, providing significantly superior performance compared to traditional Word2Vec embeddings. The marginal gains from fusion suggest that while Word2Vec adds some value, BERT's capabilities are the dominant factor.
*   Further investigation could involve exploring different fusion strategies (e.g., more complex neural network layers for combining embeddings, attention mechanisms) or experimenting with pre-trained Word2Vec models trained on larger, more domain-specific corpora to potentially amplify the benefits of the fusion approach.
